# 🍽️ Food Recipe RAG Chatbot
**Stack:** Ollama (LLM) + ChromaDB (Vector DB) + pdfplumber (PDF parsing)

This notebook builds a Retrieval-Augmented Generation (RAG) chatbot over your recipe PDFs.

**Pipeline:**
1. Install dependencies & start Ollama
2. Load & chunk recipe PDFs
3. Embed chunks → store in ChromaDB
4. Query: retrieve relevant chunks → ask Ollama → get answer

## ⚙️ Step 1 — Install System Dependencies & Start Ollama

In [1]:
%%bash
apt-get install -y zstd -q
curl -fsSL https://ollama.com/install.sh | sh

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (8,437 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [2]:
# Kill any existing Ollama processes and free port 8000
!pkill -f ollama || true
!fuser -k 8000/tcp || true
!sleep 2

# Start Ollama server in background
!nohup bash -c "OLLAMA_HOST=0.0.0.0:8000 OLLAMA_ORIGIN=* ollama serve" > /content/nohup.out 2>&1 &
!sleep 5 && tail /content/nohup.out

^C
time=2026-06-08T12:01:55.475Z level=INFO source=routes.go:1921 msg="Ollama cloud disabled: false"
time=2026-06-08T12:01:55.475Z level=INFO source=images.go:864 msg="total blobs: 0"
time=2026-06-08T12:01:55.475Z level=INFO source=images.go:871 msg="total unused blobs removed: 0"
time=2026-06-08T12:01:55.476Z level=INFO source=routes.go:1981 msg="Listening on [::]:8000 (version 0.30.6)"
time=2026-06-08T12:01:55.477Z level=INFO source=model_list_cache.go:111 msg="model list cache hydration complete" models=0 failures=0 elapsed=236.712µs
time=2026-06-08T12:01:55.477Z level=INFO source=runner.go:60 msg="discovering available GPUs..."
time=2026-06-08T12:01:55.609Z level=INFO source=model_recommendations.go:177 msg="model recommendations cache sleep scheduled" wait=3h39m40.180697458s consecutive_failures=0
time=2026-06-08T12:01:58.804Z level=WARN source=model_show_cache.go:362 msg="failed to hydrate cloud model show cache" model=nemotron-3-super error="Post \"https://ollama.com:443/api/sho

In [3]:
import os
os.environ["OLLAMA_HOST"] = "http://0.0.0.0:8000"

# Using llama3.2:3b — lightweight, fast, great for RAG on Colab free tier
# Alternatives: mistral:7b-instruct-q4_0 (smarter, needs more RAM), phi3:mini (very fast)
OLLAMA_MODEL = "llama3.2:3b"

!ollama pull {OLLAMA_MODEL}

## 📦 Step 2 — Install Python Dependencies

In [4]:
!pip install -q pdfplumber chromadb sentence-transformers langchain langchain-community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 108.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 127.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 96.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9

## 📄 Step 3 — Load & Chunk Recipe PDFs

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import pdfplumber
import os
from pathlib import Path

DATA_DIR = "/content/drive/My Drive/Colab Notebooks/NTI/foodData"

def extract_text_from_pdf(pdf_path: str) -> str:
    """Extract all text from a PDF file."""
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text

def chunk_text(text: str, chunk_size: int = 500, overlap: int = 100) -> list[str]:
    """
    Split text into overlapping chunks.
    - chunk_size: characters per chunk (500 works well for recipes)
    - overlap: shared characters between consecutive chunks (preserves context)
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

# Load all PDFs
all_chunks = []
all_metadata = []
all_ids = []

pdf_files = list(Path(DATA_DIR).glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF files:\n")

for pdf_path in pdf_files:
    print(f"  Processing: {pdf_path.name}")
    raw_text = extract_text_from_pdf(str(pdf_path))
    chunks = chunk_text(raw_text)

    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        all_metadata.append({"source": pdf_path.name, "chunk_index": i})
        all_ids.append(f"{pdf_path.stem}_chunk_{i}")

print(f"\n✅ Total chunks created: {len(all_chunks)}")

Found 11 PDF files:

  Processing: Dal Makhani Prana.pdf
  Processing: indian Vegetarian instant pot cookbook Author Archana Mundhe.pdf
  Processing: Idli Recipes Sify Food.pdf
  Processing: Murgh Pakora Crispy Indian Chicken Fritters Taz Doolittle.pdf
  Processing: trimed-america-cook-book.pdf
  Processing: Kadhi Pakora Dani Valent.pdf
  Processing: Aloo Gobi (Spicy Potato and Cauliflower) Bosch.pdf
  Processing: Paratha Global Young Academy.pdf
  Processing: Chana Masala Prana.pdf
  Processing: Tarka Dal recipe TOPdesk Careers.pdf
  Processing: Thai_Recipes.pdf

✅ Total chunks created: 686


## 🗄️ Step 4 — Create ChromaDB Vector Store

In [7]:
import chromadb
from chromadb.utils import embedding_functions

# Use a lightweight sentence-transformer model for embeddings
# all-MiniLM-L6-v2: small (80MB), fast, great quality for semantic search
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=EMBEDDING_MODEL
)

# Persistent client — data survives notebook restarts (saved to /content/chroma_db)
chroma_client = chromadb.PersistentClient(path="/content/chroma_db")

# Delete existing collection if re-running (fresh start)
try:
    chroma_client.delete_collection("food_recipes")
    print("Deleted existing collection.")
except:
    pass

collection = chroma_client.create_collection(
    name="food_recipes",
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"}  # cosine similarity for text
)

print("✅ ChromaDB collection created.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ ChromaDB collection created.


In [8]:
# Add chunks in batches (ChromaDB recommends batches of ~500)
BATCH_SIZE = 500

for i in range(0, len(all_chunks), BATCH_SIZE):
    batch_docs = all_chunks[i:i + BATCH_SIZE]
    batch_meta = all_metadata[i:i + BATCH_SIZE]
    batch_ids = all_ids[i:i + BATCH_SIZE]

    collection.add(
        documents=batch_docs,
        metadatas=batch_meta,
        ids=batch_ids
    )
    print(f"  Added batch {i // BATCH_SIZE + 1} ({len(batch_docs)} chunks)")

print(f"\n✅ All {collection.count()} chunks indexed in ChromaDB!")

  Added batch 1 (500 chunks)
  Added batch 2 (186 chunks)

✅ All 686 chunks indexed in ChromaDB!


## 🤖 Step 5 — Build the RAG Chatbot

In [9]:
import requests
import json

OLLAMA_URL = "http://localhost:8000/api/chat"

def retrieve_context(query: str, n_results: int = 5) -> tuple[str, list[str]]:
    """
    Retrieve the top-n most relevant chunks from ChromaDB.
    Returns: (formatted context string, list of source filenames)
    """
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )

    docs = results["documents"][0]
    metas = results["metadatas"][0]

    context_parts = []
    sources = []
    for doc, meta in zip(docs, metas):
        context_parts.append(f"[Source: {meta['source']}]\n{doc}")
        sources.append(meta["source"])

    return "\n\n---\n\n".join(context_parts), list(set(sources))


def ask_ollama(messages: list[dict]) -> str:
    """Send a chat request to Ollama and return the response text."""
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": OLLAMA_MODEL,
            "stream": False,
            "messages": messages
        }
    )
    response.raise_for_status()
    return response.json()["message"]["content"]


SYSTEM_PROMPT = """You are a helpful cooking assistant. You answer questions about food recipes
based ONLY on the context provided below. If the answer is not in the context, say so honestly.
Be concise, friendly, and practical. When listing ingredients or steps, use a clear format."""


def rag_query(user_question: str, chat_history: list[dict] = None, n_results: int = 5) -> dict:
    """
    Full RAG pipeline:
    1. Retrieve relevant context from ChromaDB
    2. Build prompt with context + chat history
    3. Get response from Ollama

    Returns dict with 'answer', 'sources', and updated 'history'.
    """
    if chat_history is None:
        chat_history = []

    # Retrieve
    context, sources = retrieve_context(user_question, n_results=n_results)

    # Build messages: system + history + new user message with context
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages.extend(chat_history)
    messages.append({
        "role": "user",
        "content": f"Context from recipe books:\n\n{context}\n\n---\n\nQuestion: {user_question}"
    })

    # Generate
    answer = ask_ollama(messages)

    # Update history (store clean question, not the context-stuffed version)
    updated_history = chat_history + [
        {"role": "user", "content": user_question},
        {"role": "assistant", "content": answer}
    ]

    return {
        "answer": answer,
        "sources": sources,
        "history": updated_history
    }

print("✅ RAG pipeline ready!")

✅ RAG pipeline ready!


## 💬 Step 6 — Chat with Your Recipe Bot

In [10]:
# Single query example
result = rag_query("What ingredients do I need to make pasta carbonara?")

print("🤖 Answer:")
print(result["answer"])
print(f"\n📚 Sources: {', '.join(result['sources'])}")

🤖 Answer:
I think there might be a bit of confusion! The text doesn't mention pasta carbonara at all. However, based on the recipe for sauce in "trimed-america-cook-book.pdf", it does provide information on making a tomato sauce that's similar to what you'd find in many Italian recipes.

If you're looking for a recipe for traditional spaghetti carbonara, I'd be happy to help you with that! However, please note that the text doesn't have any specific ingredients listed for pasta carbonara. If you could provide me with more context or information on how to make it, I'll do my best to assist you.

📚 Sources: Thai_Recipes.pdf, indian Vegetarian instant pot cookbook Author Archana Mundhe.pdf, trimed-america-cook-book.pdf, Paratha Global Young Academy.pdf


In [11]:
# Multi-turn conversation example
history = []

questions = [
    "What desserts are in the recipes?",
    "Which of those requires the least sugar?",
    "Give me the step-by-step instructions for that one."
]

for q in questions:
    print(f"\n👤 You: {q}")
    result = rag_query(q, chat_history=history)
    print(f"🤖 Bot: {result['answer']}")
    print(f"   📚 Sources: {', '.join(result['sources'])}")
    history = result["history"]  # Carry history forward


👤 You: What desserts are in the recipes?
🤖 Bot: Unfortunately, there is no dessert mentioned in any of the provided recipe sources. The recipes provided cover a variety of dishes including Roundsteak Breasteau, Batata Bhaji, Cha Om Omelet, and more Indian Vegetarian Instant Pot recipes. If you're looking for desserts, I'd be happy to help you find one!
   📚 Sources: Thai_Recipes.pdf, indian Vegetarian instant pot cookbook Author Archana Mundhe.pdf, trimed-america-cook-book.pdf

👤 You: Which of those requires the least sugar?
🤖 Bot: Based on the recipes provided, the Nut Bread recipe by Brig.-General Malin Craig has only 2 teaspoons of sugar out of a total list of ingredients that doesn't contain much in terms of sweet ingredients except for brown sugar and molasses. 

However, another option is also Salad Dressing for Lettuce or Tomatoes, which calls for only 4 tablespoons (or 1/8 cup) of sugar out of the entire ingredient list.
   📚 Sources: indian Vegetarian instant pot cookbook Aut

## 🖥️ Step 7 — Interactive Chat Loop (Optional)

In [12]:
# Run this cell for an interactive terminal-style chat
# Type 'quit' or 'exit' to stop

print("🍽️ Food Recipe Chatbot — Ask me anything about the recipes!")
print("   Type 'quit' to exit, 'reset' to clear history\n")

chat_history = []

while True:
    try:
        user_input = input("You: ").strip()
    except EOFError:
        break

    if not user_input:
        continue
    if user_input.lower() in ("quit", "exit"):
        print("Goodbye! 👋")
        break
    if user_input.lower() == "reset":
        chat_history = []
        print("🔄 Chat history cleared.\n")
        continue

    result = rag_query(user_input, chat_history=chat_history)
    print(f"\n🤖 Bot: {result['answer']}")
    print(f"   📚 Sources: {', '.join(result['sources'])}\n")
    chat_history = result["history"]

🍽️ Food Recipe Chatbot — Ask me anything about the recipes!
   Type 'quit' to exit, 'reset' to clear history

You: Give me Chana Masala recepie

🤖 Bot: Here's the recipe for Chana Masala:

**Ingredients:**

* 1. Heat the oil in a pan and add the cumin seeds and onions (no amount mentioned)
* 2 cups pre-soaked and cooked chickpeas or 2 cans chickpea
* 1 tsp red chilli powder
* 1/2 tsp cumin powder
* 1 tsp garam masala powder
* 1 tsp coriander powder
* 1/2 cup water or light coconut milk
* Salt to taste

**Instructions:**

1. Heat the oil in a pan and add the cumin seeds.
2. Add onions and cook until they are soft (no amount mentioned).
3. Add the chickpeas, red chilli powder, cumin powder, garam masala powder, coriander powder, salt, and water or coconut milk. Mix everything together.
4. Cook for up to 10 minutes before tasting and adding more salt if needed.
5. Garnish with chopped coriander, sliced raw onion, and a slice of lime.

Note: There is no amount mentioned for the onions, so 

## 🔧 Utilities — Inspect & Debug

In [13]:
# See what's stored in ChromaDB
print(f"Total chunks in DB: {collection.count()}")

# Peek at a few records
sample = collection.peek(limit=3)
for i, (doc, meta) in enumerate(zip(sample["documents"], sample["metadatas"])):
    print(f"\n--- Chunk {i+1} (from {meta['source']}) ---")
    print(doc[:300] + "..." if len(doc) > 300 else doc)

Total chunks in DB: 686

--- Chunk 1 (from Dal Makhani Prana.pdf) ---
The tastes and aromas
Dal Makhani
of Indian food by
Saroj Velho
Dal Makhani is one of North India’s most
characteristic dishes. It is a lentil dish high in
protein - an important attribute in primarily
vegetarian cultures. Lentils contain dietary fibre,
folate (vitamin B) and iron which is good for ...

--- Chunk 2 (from Dal Makhani Prana.pdf) ---
ncreased. Lentils, mixed with grains such as rice,
are a complete protein dish. Enjoy with pulao rice,
naan bread or toasted Turkish bread.
Ingredients Let’s Create
• 1 cup lentils and 1/2 cup red 1. For both canned and pre-soaked: drain and rinse
kidney thoroughly before simmering with a little sal...

--- Chunk 3 (from Dal Makhani Prana.pdf) ---
dd yoghurt and cream and set aside.
yogurt and fresh
• cream 3. In a wok heat oil, splutter cumin and coriander seeds,
• 2 tbsp vegetable oil curry leaves. Add garlic paste, fry for few seconds,
• 1 tsp coriander seeds then add on

In [14]:
# Test retrieval for a specific query — see what context gets pulled
test_query = "chocolate cake"
context, sources = retrieve_context(test_query, n_results=3)

print(f"Top chunks for query: '{test_query}'\n")
print(context[:1000])  # Print first 1000 chars

Top chunks for query: 'chocolate cake'

[Source: trimed-america-cook-book.pdf]
OK
46
U. S. SENATOR F. B. WILLIS, State of Ohio
Roundsteak Breasteau
Make a cream sauce of 1 tablespoon each of flour and
butter until smooth, add 1 pt. of white stock, or milk; cook
to smooth gTavy. To the sauce add 1 pt. of ground steak, 3
cups of bread crumbs, salt and pepper to taste. At last
moment add 3 eggs beaten to a froth. Bake in buttered muffin
tins set in cold water. Cover with paraffin paper. Bake 20
minutes. Serve with more sauce. ]\Iakes 12 cakes.
CONGRESSMAN E. 0. LEATHERWOOD, S

---

[Source: trimed-america-cook-book.pdf]
shortening,
1 cup of raisins and 1 teaspoon of salt. Dissolve yeast cake
and 1 tablespoon of com syrup in lukewarm water add 1 cup
;
of flour and milk, potatoes, shortening and com syrup well
creamed. Cover and set in a warm place to rise. When light,
add raisins that have been well floured, and salt. Knead
lightly and let rise again until double its bulk. Mold into
loaves

In [15]:
# Check Ollama is responding
import requests
r = requests.get("http://localhost:8000/api/tags")
models = [m["name"] for m in r.json().get("models", [])]
print(f"Ollama is running. Available models: {models}")

Ollama is running. Available models: ['llama3.2:3b']
